# Bloque 4 — Pandas avanzado
## Series · Variables categóricas · Agrupaciones · Combinaciones · Parquet
## Proyecto: Red Social Ficticia

En el Bloque 3 cargamos, inspeccionamos y limpiamos los datos. Ahora **extraemos valor** de ellos: trabajamos con columnas sueltas (Series), optimizamos y codificamos variables categóricas, resumimos por grupos, combinamos tablas y guardamos el resultado en el formato **Parquet**.

**Contenidos:** Series (índice, posición y etiqueta), tipo `category` y memoria, codificación one-hot y ordinal, `groupby` con `agg`, `pivot_table`, `crosstab`, `merge` y el formato Parquet.

> Los gráficos se trabajan en un bloque posterior, así que aquí no se pide ninguno.
> Este bloque genera el fichero `../data/dataset_procesado.parquet`, que se usará en el Bloque 5.

---
### Objetivos
- Crear Series a partir de columnas y operar con ellas: propiedades, orden, nulos, frecuencias y selección por posición y por etiqueta.
- Convertir columnas de texto a `category`, medir el ahorro de memoria y codificarlas (one-hot y ordinal).
- Resumir datos por grupos, con tablas dinámicas y de contingencia.
- Combinar tablas con `merge` y validar las claves.
- Guardar y leer datos en formato Parquet, y comparar sus ventajas con el CSV.

### Cómo trabajar este cuaderno
1. Ejecuta primero la celda de carga de datos.
2. Completa los **ejercicios básicos** en orden: cada celda indica el nombre de las variables que debes crear. Sustituye cada `None` (o `pass`) por tu código. Los ejercicios se apoyan unos en otros.
3. Los enunciados explican **qué** se pide y muestran ejemplos del resultado con datos inventados, pero no indican qué métodos usar: búscalos en tus apuntes y en las listas de ejercicios. La única excepción es Parquet, que no se ha visto antes y se explica en este cuaderno.
4. Continúa con los **ejercicios avanzados** y, para terminar, implementa las funciones de la sección "Entrega" en `../scripts/`.

### Relación con los apuntes de Pandas

Este bloque reúne contenidos que en los apuntes aparecen repartidos en varios cuadernos. La parte de **Series** conecta con el cuaderno de Series; las **variables categóricas** conectan con el cuaderno de `category`, `CategoricalDtype`, memoria y orden lógico; y las **agrupaciones y combinaciones** conectan con `groupby`, `agg`, `pivot_table`, `crosstab` y `merge`.

La idea no es memorizar una instrucción aislada, sino reconocer el tipo de problema:
- Si trabajas con una sola columna y su índice, estás usando una **Serie**.
- Si una columna de texto repite pocos valores, puede convenir convertirla a **categórica**.
- Si quieres resumir filas por grupos, necesitas **agrupaciones** o **tablas dinámicas**.
- Si tienes información repartida entre varias tablas, necesitas una **combinación** con claves.
- Si quieres guardar el resultado final conservando tipos y reduciendo tamaño, puedes usar **Parquet**.

El único contenido que se introduce aquí de forma nueva es **Parquet**. Por eso se explica dentro del propio cuaderno antes de pedir el ejercicio correspondiente.


## 1. Carga de datos

In [ ]:
# Cargamos las tablas del proyecto y aplicamos la limpieza de nulos que ya
# vimos en el Bloque 3. Esta celda ya está resuelta, no la modifiques.
import os
import pandas as pd

df_usuarios = pd.read_csv('../data/usuarios.csv')
df_publicaciones = pd.read_excel('../data/publicaciones.xlsx', sheet_name='publicaciones')

df_usuarios['edad'] = df_usuarios['edad'].fillna(df_usuarios['edad'].median())
df_usuarios['ciudad'] = df_usuarios['ciudad'].fillna('Desconocido')
df_usuarios['bio'] = df_usuarios['bio'].fillna('Desconocido')
df_publicaciones['likes'] = df_publicaciones['likes'].fillna(df_publicaciones['likes'].median())
df_publicaciones['comentarios'] = df_publicaciones['comentarios'].fillna(df_publicaciones['comentarios'].median())

print('usuarios:', df_usuarios.shape)
print('publicaciones:', df_publicaciones.shape)


---
## Ejercicios básicos

Completa las siguientes 8 celdas trabajando siempre con los datos reales de `../data/`. Los ejemplos de cada enunciado usan datos inventados y **nunca** son la solución: solo muestran el formato del resultado esperado. Los ejercicios son consecutivos: los últimos reutilizan tablas creadas en los primeros.

In [ ]:
# ─── Ejercicio 1 — Series: creación, propiedades, orden y selección ───────
# Una Serie es una columna con su propio índice: cada valor tiene una
# etiqueta. Trabaja con 'df_usuarios'.
#
# 1. Guarda en 'seguidores_por_nombre' una Serie con el número de seguidores
#    de cada usuario, en la que el ÍNDICE sea el nombre del usuario y el
#    VALOR su número de seguidores.
#    Por ejemplo, con estos datos (esto NO es la solución), la Serie se
#    vería así:
#      Ana        5000
#      Borja      1200
#      Carmen      800
# 2. Guarda en 'n_elementos' cuántos elementos tiene la Serie, en
#    'tipo_dato' su tipo de dato escrito como texto y en 'primera_etiqueta'
#    la primera etiqueta de su índice.
# 3. Guarda en 'seguidores_ordenados' la Serie ordenada por valor, de MAYOR
#    a MENOR, y en 'top_3_seguidores' sus 3 primeros elementos.
# 4. Guarda en 'primer_valor_posicion' el valor que ocupa la posición 0 de
#    'seguidores_por_nombre' (seleccionado por POSICIÓN) y en
#    'valor_beatriz' el valor de la etiqueta 'Beatriz' (seleccionado por
#    ETIQUETA; en estos datos solo hay una usuaria con ese nombre).

seguidores_por_nombre = None
n_elementos = None
tipo_dato = None
primera_etiqueta = None
seguidores_ordenados = None
top_3_seguidores = None
primer_valor_posicion = None
valor_beatriz = None


In [ ]:
# ─── Ejercicio 2 — Series: nulos, operaciones y frecuencias ─────────────────
# 1. Guarda en 'likes_sin_limpiar' la columna 'likes' de las publicaciones
#    tal como está en el fichero '../data/publicaciones.xlsx' (hoja
#    'publicaciones'), sin rellenar sus nulos. Tienes que volver a leerla,
#    porque 'df_publicaciones' ya tiene los nulos rellenos.
# 2. Guarda en 'n_nulos_likes' cuántos nulos tiene esa columna y en
#    'media_likes' su media ignorando los nulos.
# 3. Guarda en 'likes_ajustados' una Serie con los likes de cada
#    publicación aumentados en 100 (un ajuste ficticio) y en
#    'primeros_5_ajustados' sus 5 primeros valores. Los nulos siguen siendo
#    nulos.
#    Por ejemplo, para los likes 10, nulo y 30 el RESULTADO sería 110, nulo
#    y 130 (esto NO es la solución).
# 4. Guarda en 'es_popular' una Serie de verdadero/falso que indique, para
#    cada usuario de 'df_usuarios', si tiene 1000 seguidores o más, y en
#    'n_populares' cuántos usuarios cumplen la condición.
# 5. Guarda en 'frecuencia_tipos' cuántos usuarios hay de cada
#    'tipo_cuenta' y en 'porcentaje_tipos' el porcentaje que representa
#    cada tipo (de 0 a 100, con 1 decimal).
#    Por ejemplo, para los valores free, free, premium y free la frecuencia
#    sería free 3 y premium 1, y los porcentajes free 75.0 y premium 25.0.

likes_sin_limpiar = None
n_nulos_likes = None
media_likes = None
likes_ajustados = None
primeros_5_ajustados = None
es_popular = None
n_populares = None
frecuencia_tipos = None
porcentaje_tipos = None


In [ ]:
# ─── Ejercicio 3 — Variables categóricas: conversión y memoria ──────────────
# Una columna de texto con pocos valores distintos que se repiten mucho
# puede guardarse como variable CATEGÓRICA: en lugar de repetir el texto en
# cada fila, Pandas guarda una lista de categorías y, en cada fila, una
# referencia a ella. Así ocupa menos memoria.
#
# 1. Guarda en 'df_cat' una COPIA de 'df_usuarios' en la que las columnas
#    'tipo_cuenta', 'genero' y 'ciudad' sean de tipo categórico.
# 2. Guarda en 'memoria_antes' la memoria que ocupa la columna 'ciudad' de
#    'df_usuarios' y en 'memoria_despues' la de 'df_cat', ambas en bytes y
#    contando el contenido real de los textos (si no se cuenta, el ahorro
#    no se ve). Guarda en 'ahorro_pct' el ahorro en porcentaje, con 1
#    decimal.
#    Por ejemplo, si una columna pasa de ocupar 1000 bytes a 250, el ahorro
#    es 75.0 (esto NO es la solución).
# 3. Guarda en 'categorias_tipo_cuenta' la lista de categorías de la
#    columna 'tipo_cuenta' de 'df_cat' y en 'n_categorias_ciudad' cuántas
#    categorías tiene la columna 'ciudad'.

df_cat = None
memoria_antes = None
memoria_despues = None
ahorro_pct = None
categorias_tipo_cuenta = None
n_categorias_ciudad = None


### Antes del ejercicio 4: one-hot frente a ordinal

Convertir una columna a `category` sirve para representar mejor una variable cualitativa dentro de Pandas. Pero si esa variable va a entrar en cálculos, modelos o comparaciones, a menudo necesitamos transformarla de otra manera.

**Codificación one-hot**: se usa cuando las categorías no tienen orden natural. Por ejemplo, `free`, `premium` y `creator` podrían convertirse en tres columnas booleanas: una indica si la fila es `free`, otra si es `premium` y otra si es `creator`. En Pandas se suele hacer con `pd.get_dummies(columna, prefix='nombre')`. Es útil porque no inventa una jerarquía entre categorías.

**Codificación ordinal**: se usa cuando sí existe un orden lógico. En este proyecto podemos interpretar `free < premium < creator`. Para eso conviene definir un `pd.CategoricalDtype(categories=[...], ordered=True)` y aplicarlo con `.astype(...)`. A partir de ahí Pandas puede ordenar, comparar, calcular mínimos y máximos categóricos, etc.

Regla práctica: si las categorías son nombres sin jerarquía, usa one-hot; si representan niveles ordenados, usa una categórica ordinal.


In [ ]:
# ─── Ejercicio 4 — Codificación: one-hot y ordinal ──────────────────────────
# Muchos algoritmos solo entienden números. Hay dos formas habituales de
# codificar una categoría: ONE-HOT (una columna de verdadero/falso por cada
# categoría, sin ningún orden entre ellas) y ORDINAL (las categorías tienen
# un orden lógico y se pueden comparar).
#
# 1. Guarda en 'dummies_tipo' la codificación one-hot de la columna
#    'tipo_cuenta' de 'df_cat': una columna por categoría, llamada
#    'tipo_cuenta_<categoría>' (100 filas y 3 columnas).
#    Por ejemplo, para los valores free, premium y free, las columnas
#    resultantes serían (esto NO es la solución):
#      tipo_cuenta_free  tipo_cuenta_premium
#                  True                False
#                 False                 True
#                  True                False
# 2. Guarda en 'usuarios_onehot' la tabla 'df_cat' SIN la columna
#    'tipo_cuenta' y con las columnas de 'dummies_tipo' añadidas.
# 3. Guarda en 'df_ordinal' una COPIA de 'df_cat' en la que 'tipo_cuenta'
#    sea una categoría ORDENADA con este orden lógico:
#    free < premium < creator.
# 4. Trabajando con 'df_ordinal', guarda en 'cuenta_min' y 'cuenta_max' la
#    categoría menor y la mayor presentes en los datos, en
#    'n_por_encima_de_free' cuántos usuarios tienen una cuenta superior a
#    'free', y en 'primera_cuenta' el 'tipo_cuenta' de la primera fila
#    después de ordenar la tabla por esa columna según el orden lógico.

dummies_tipo = None
usuarios_onehot = None
df_ordinal = None
cuenta_min = None
cuenta_max = None
n_por_encima_de_free = None
primera_cuenta = None


In [ ]:
# ─── Ejercicio 5 — Agrupaciones, tablas dinámicas y de contingencia ─────────
# 1. Guarda en 'resumen_por_cuenta' un resumen de 'df_ordinal' por
#    'tipo_cuenta' con tres columnas: 'media_seguidores', 'max_seguidores' y
#    'n_usuarios'. Solo deben aparecer las categorías presentes, en el
#    orden lógico free, premium, creator.
#    Por ejemplo, con otros datos, el RESULTADO se vería así (esto NO es la
#    solución):
#               media_seguidores  max_seguidores  n_usuarios
#      free                800.0            1500           3
#      premium            2500.0            4000           2
# 2. Guarda en 'likes_por_tipo' los likes totales de cada 'tipo_contenido'
#    de 'df_publicaciones', ordenados de mayor a menor.
# 3. Crea 'pubs_mes' como COPIA de 'df_publicaciones' con una nueva columna
#    'mes' que contenga el número de mes (de 1 a 12) de cada publicación, a
#    partir de la columna 'fecha' (formato 'AAAA-MM-DD'). Guarda en
#    'tabla_mes' una tabla dinámica con 'tipo_contenido' en las filas, 'mes'
#    en las columnas y la MEDIA de likes como valor (5 filas y 12 columnas).
# 4. Guarda en 'cuentas_por_genero' una tabla de contingencia con el número
#    de usuarios de cada combinación de 'tipo_cuenta' (filas) y 'genero'
#    (columnas) de 'df_usuarios'.
#    Por ejemplo, con 3 usuarios (free-F, free-M y premium-F) el RESULTADO
#    sería:
#               F  M
#      free     1  1
#      premium  1  0

resumen_por_cuenta = None
likes_por_tipo = None
pubs_mes = None
tabla_mes = None
cuentas_por_genero = None


In [ ]:
# ─── Ejercicio 6 — Combinación de tablas y engagement ───────────────────────
# 1. Guarda en 'df_merged' la tabla que resulta de combinar
#    'df_publicaciones' con 'df_usuarios' mediante 'id_usuario',
#    conservando TODAS las publicaciones aunque su autor no apareciera en
#    la tabla de usuarios. Cada fila tendrá los datos de la publicación y
#    los de su autor.
#    Por ejemplo, con estas dos tablas (esto NO es la solución):
#      publicaciones                usuarios
#      id_publicacion  id_usuario   id_usuario  nombre
#                   1           1            1     Ana
#                   2           9            2   Borja
#    la combinación daría:
#      id_publicacion  id_usuario  nombre
#                   1           1     Ana
#                   2           9     (nulo)
# 2. Guarda en 'filas_merged' el número de filas de 'df_merged' (debe ser
#    el mismo que el de 'df_publicaciones', 300).
# 3. Añade a 'df_merged' una columna 'engagement_rate' con la fórmula:
#      (likes + comentarios + compartidos) / visualizaciones
#    Si una publicación tiene 0 visualizaciones, su engagement debe ser 0
#    (nunca nulo ni infinito).
#    Por ejemplo, una publicación con 100 likes, 20 comentarios, 10
#    compartidos y 500 visualizaciones tiene un engagement de 0.26.

df_merged = None
filas_merged = None


In [ ]:
# ─── Ejercicio 7 — Rankings y resumen por ciudad ────────────────────────────
# Trabaja con 'df_merged' (ejercicio 6, ya con 'engagement_rate').
#
# 1. Guarda en 'top_10_creadores' una tabla con UNA fila por usuario y su
#    engagement medio, ordenada de mayor a menor y con solo los 10
#    primeros. Debe tener las columnas 'id_usuario', 'nombre',
#    'tipo_cuenta' y 'engagement_promedio'. Ojo: hay nombres repetidos, así
#    que un usuario se identifica por su 'id_usuario', no solo por el
#    nombre.
#    Por ejemplo, con otros datos (esto NO es la solución):
#      id_usuario  nombre  tipo_cuenta  engagement_promedio
#              17   Carlos      premium                 0.90
#              35    Elena         free                 0.40
# 2. Guarda en 'resumen_ciudades' una tabla con una fila por ciudad (la
#    ciudad es el índice) y tres columnas: 'media_seguidores' (media de la
#    columna 'seguidores' de las filas de esa ciudad), 'n_publicaciones' y
#    'engagement_promedio'. Ordénala por 'n_publicaciones', de mayor a
#    menor.
# 3. Guarda en 'ciudad_mas_engagement' el nombre de la ciudad con mayor
#    'engagement_promedio'.

top_10_creadores = None
resumen_ciudades = None
ciudad_mas_engagement = None


---
## El formato Parquet

Este formato no aparece en los cuadernos de teoría anteriores, así que se explica aquí antes del último ejercicio.

### ¿Qué es?
**Apache Parquet** es un formato de fichero **binario y columnar** diseñado para el análisis de datos. Es el estándar en los pipelines de datos actuales (Spark, servicios en la nube, almacenes de datos...). Un fichero Parquet no se puede abrir con un editor de texto: se lee con librerías, como Pandas.

### Diferencias con el CSV
| | CSV | Parquet |
|---|---|---|
| Contenido | texto plano | binario |
| Organización | por **filas** | por **columnas** |
| Tipos de datos | no los guarda: al leer hay que volver a interpretarlos (fechas, categorías...) | **los conserva** (enteros, decimales, booleanos, fechas y categóricas) |
| Tamaño | grande | **comprimido**, normalmente mucho menor |
| Lectura | hay que leer todo el fichero | permite **leer solo las columnas** que necesitas |

### Compresión
Parquet comprime cada columna por separado. El códec más habitual es **snappy**: comprime y descomprime muy rápido con una reducción de tamaño moderada. Otros códecs, como **gzip**, comprimen más pero son más lentos.

### Cómo se usa desde Pandas
| Operación | Instrucción |
|---|---|
| Guardar | `df.to_parquet(ruta, compression='snappy', index=False)` |
| Leer todo | `pd.read_parquet(ruta)` |
| Leer solo algunas columnas | `pd.read_parquet(ruta, columns=['col_1', 'col_2'])` |

Detalles que conviene conocer:
- Necesita un **motor** instalado: `pyarrow` (el que usa Pandas por defecto) o `fastparquet`. Ya están instalados en este proyecto.
- **No crea carpetas**: si la carpeta de destino no existe, hay que crearla antes (por ejemplo, con `os.makedirs(carpeta, exist_ok=True)`).
- Por defecto guarda también el índice; con `index=False` se omite.
- Los nombres de las columnas deben ser texto.
- Con tablas muy pequeñas el ahorro de tamaño puede ser mínimo o incluso nulo, porque el fichero incluye metadatos.

La celda siguiente es una demostración ya resuelta con datos inventados. Escribe en memoria (no crea ficheros) para que veas cómo se conservan los tipos y cómo se leen solo unas columnas.

In [ ]:
# Demostración de Parquet con datos inventados (ya resuelta, solo para ejecutar).
import io

demo = pd.DataFrame({
    'id': [1, 2, 3],
    'tipo': pd.Categorical(['a', 'b', 'a']),
    'valor': [0.5, 1.5, 2.5],
})

buffer = io.BytesIO()                       # un "fichero" en memoria
demo.to_parquet(buffer, compression='snappy', index=False)
print('Bytes escritos:', len(buffer.getvalue()))

buffer.seek(0)
leido = pd.read_parquet(buffer)
print('\nTipos tras leer (la columna categórica sigue siéndolo):')
print(leido.dtypes)

buffer.seek(0)
solo_dos = pd.read_parquet(buffer, columns=['id', 'valor'])
print('\nLectura de solo dos columnas:', list(solo_dos.columns))


In [ ]:
# ─── Ejercicio 8 — Guardar y leer en Parquet, y comparar con CSV ──────────
# Vas a guardar la tabla final del proyecto ('df_merged', con
# 'engagement_rate') en Parquet y a comprobar sus ventajas frente al CSV.
# Apóyate en la explicación anterior.
#
# 1. Guarda 'df_merged' en '../data/dataset_procesado.parquet' con
#    compresión snappy y sin incluir el índice.
# 2. Lee de vuelta ese fichero y guarda la tabla en 'df_parquet'. Guarda en
#    'misma_forma' un valor True/False que indique si tiene las mismas
#    filas y columnas que 'df_merged'.
# 3. Guarda en 'df_columnas' solo las columnas 'nombre', 'tipo_cuenta' y
#    'engagement_rate' del fichero, leyendo únicamente esas tres columnas
#    (300 filas y 3 columnas).
# 4. Guarda también 'df_merged' en el fichero CSV
#    '../data/dataset_procesado.csv' (sin índice). Guarda en
#    'tamano_parquet' y 'tamano_csv' el tamaño en bytes de cada fichero (el
#    módulo estándar 'os' permite consultarlo) y en 'parquet_es_menor' un
#    valor True/False que indique si el Parquet ocupa menos que el CSV.
#    Por ejemplo, un fichero de 34000 bytes frente a otro de 61000 daría
#    True (esto NO es la solución, tus tamaños serán otros).

df_parquet = None
misma_forma = None
df_columnas = None
tamano_parquet = None
tamano_csv = None
parquet_es_menor = None


### Entrega — `../scripts/pipeline.py`

Implementa estas funciones en `../scripts/pipeline.py`. Cada una está vinculada a uno de los ejercicios básicos anteriores:

| Función | Descripción |
|---|---|
| `calcular_engagement(df)` | Devuelve una copia del DataFrame con la columna `engagement_rate` = (`likes` + `comentarios` + `compartidos`) / `visualizaciones`, que vale 0 cuando `visualizaciones` es 0 (ejercicio 6) |
| `top_creadores(df_merged, n=10)` | Devuelve un DataFrame con una fila por usuario (`id_usuario`, `nombre`, `tipo_cuenta`, `engagement_promedio`), ordenado de mayor a menor y limitado a `n` filas (ejercicio 7) |
| `resumen_por_ciudad(df_merged)` | Devuelve un DataFrame indexado por ciudad con `media_seguidores`, `n_publicaciones` y `engagement_promedio` (ejercicio 7) |
| `guardar_parquet(df, ruta)` | Guarda el DataFrame en Parquet con compresión snappy, crea las carpetas intermedias que falten y devuelve `None` (ejercicio 8) |

Comprueba tu implementación con:

```bash
python -m pytest ../tests/test_bloque4.py -v
```

---
## Ejercicios avanzados

Estos ejercicios exigen decidir cuándo compensa una optimización, validar las claves de una combinación y comparar formatos de almacenamiento. Trabajan con los datos reales de `../data/` y con la tabla `df_merged` (con `engagement_rate`) del ejercicio 6. Igual que en los básicos, los enunciados no indican qué métodos usar y los ejemplos usan datos inventados: nunca son la solución.

In [ ]:
# ─── Ejercicio A.1 — Cuándo compensa una variable categórica ────────────────
# Convertir a categórica no siempre compensa: depende de cuántos valores
# distintos haya y de cuánto se repitan.
#
# 1. Define una función 'analizar_categorias(df, columnas)' que reciba una
#    tabla y una lista de nombres de columnas de texto, y devuelva una
#    NUEVA tabla con una fila por columna (el ÍNDICE es el nombre de la
#    columna) y estas columnas:
#      'tipo'                  -> tipo de dato de la columna tras
#                                 convertirla a categórica, como texto
#      'categorias'            -> número de categorías
#      'memoria_antes_bytes'   -> memoria de la columna original, contando
#                                 el contenido de los textos
#      'memoria_despues_bytes' -> memoria de la columna ya categórica
#      'ahorro_bytes'          -> memoria antes menos memoria después
#    La función NO debe modificar la tabla original.
#    Por ejemplo, para una tabla con las columnas 'tipo' (free, premium,
#    free) y 'ciudad' (Madrid, Bilbao, Madrid), el RESULTADO tendría dos
#    filas ('tipo' y 'ciudad'), las dos con tipo 'category' y 2 categorías
#    (las cifras de memoria dependen de cada máquina).
# 2. Guarda en 'auditoria_usuarios' el resultado de analizar las columnas
#    'tipo_cuenta', 'genero', 'ciudad' y 'pais' de 'df_usuarios'.
# 3. Guarda en 'auditoria_texto_libre' el resultado de analizar las
#    columnas 'nombre', 'apellidos' y 'bio' de 'df_usuarios'.
# 4. Guarda en 'columna_mayor_ahorro' el nombre de la columna de
#    'auditoria_usuarios' con mayor ahorro en bytes.
# 5. Escribe en un comentario, debajo de las variables, en cuáles de las
#    columnas analizadas conviene usar categóricas y en cuáles no, y por
#    qué (fíjate en cuántas categorías tiene cada una respecto al número de
#    filas).

def analizar_categorias(df, columnas):
    pass  # Tu código aquí


auditoria_usuarios = None
auditoria_texto_libre = None
columna_mayor_ahorro = None

# Tu razonamiento:


In [ ]:
# ─── Ejercicio A.2 — Validación de claves al combinar tablas ────────────────
# Antes de combinar tablas hay que comprobar que la clave se comporta como
# esperamos; si no, se pueden duplicar o perder filas sin darse cuenta.
#
# 1. Guarda en 'usuarios_unicos' un valor True/False que indique si cada
#    'id_usuario' aparece una sola vez en 'df_usuarios'.
# 2. Combina 'df_publicaciones' con 'df_usuarios' por 'id_usuario',
#    conservando todas las publicaciones y pidiendo a Pandas que añada una
#    columna que indique de qué tabla procede cada fila. Guarda el
#    resultado en 'merge_con_origen' y, a partir de esa columna, guarda en
#    'n_sin_usuario' cuántas publicaciones tienen un usuario que no existe
#    en 'df_usuarios'.
#    Por ejemplo, si de 4 publicaciones hay 1 cuyo 'id_usuario' no existe
#    entre los usuarios, el RESULTADO de 'n_sin_usuario' sería 1.
# 3. Repite la combinación pidiendo a Pandas que compruebe la relación
#    entre las tablas: MUCHAS publicaciones para UN usuario. Guarda el
#    resultado en 'merge_validado' (debe conservar las 300 filas).
# 4. Repite la combinación exigiendo ahora una relación UNO a UNO, que en
#    estos datos NO se cumple. Captura el error con try/except y guarda en
#    'tipo_error' el nombre del tipo de error que se produce, como texto.
# 5. Guarda en 'usuarios_sin_publicaciones' cuántos usuarios de
#    'df_usuarios' no aparecen en ninguna publicación (compara las dos
#    tablas usando de nuevo la columna que indica el origen).

usuarios_unicos = None
merge_con_origen = None
n_sin_usuario = None
merge_validado = None
tipo_error = None
usuarios_sin_publicaciones = None


In [ ]:
# ─── Ejercicio A.3 — Tabla dinámica de engagement ───────────────────────────
# 1. Define una función 'tabla_engagement_ciudad_contenido(df)' que reciba
#    una tabla con las columnas 'tipo_contenido', 'ciudad' y
#    'engagement_rate' y devuelva una tabla dinámica con 'tipo_contenido' en
#    las filas, 'ciudad' en las columnas y el engagement MEDIO como valor.
#    Las combinaciones sin ninguna publicación deben aparecer con 0.
#    Por ejemplo, con estas publicaciones (esto NO es la solución):
#      tipo_contenido  ciudad  engagement_rate
#      foto            Madrid              0.2
#      foto            Madrid              0.4
#      video           Bilbao              0.8
#    el RESULTADO sería:
#             Bilbao  Madrid
#      foto      0.0     0.3
#      video     0.8     0.0
# 2. Guarda en 'tabla_engagement' el resultado de aplicarla a 'df_merged'.
# 3. Guarda en 'mejor_combinacion' la pareja (tipo_contenido, ciudad) con
#    mayor engagement medio de esa tabla (pista: reorganiza la tabla a
#    formato largo, con una fila por pareja).

def tabla_engagement_ciudad_contenido(df):
    pass  # Tu código aquí


tabla_engagement = None
mejor_combinacion = None


In [ ]:
# ─── Ejercicio A.4 — Comparación de formatos: CSV frente a Parquet ──────────
# 1. Define una función 'comparar_formatos(df, ruta_base)' que guarde la
#    tabla en dos ficheros, '<ruta_base>.csv' y '<ruta_base>.parquet'
#    (el CSV sin índice y el Parquet con compresión snappy), y devuelva un
#    diccionario con:
#      'csv_bytes'     -> tamaño en bytes del fichero CSV
#      'parquet_bytes' -> tamaño en bytes del fichero Parquet
#      'ratio'         -> parquet_bytes / csv_bytes, con 3 decimales
#    Por ejemplo, para ficheros de 61000 y 34000 bytes el RESULTADO sería
#    {'csv_bytes': 61000, 'parquet_bytes': 34000, 'ratio': 0.557} (esto NO
#    es la solución).
# 2. Guarda en 'comparacion' el resultado de aplicarla a 'df_merged' con la
#    ruta base '../data/comparativa_formatos'.
# 3. Guarda en 'parquet_mas_pequeno' un valor True/False que indique si el
#    Parquet ocupa menos que el CSV.
# 4. Escribe en un comentario por qué, con una tabla muy pequeña (por
#    ejemplo, de 2 filas), el Parquet podría NO ocupar menos que el CSV.

def comparar_formatos(df, ruta_base):
    pass  # Tu código aquí


comparacion = None
parquet_mas_pequeno = None

# Tu razonamiento:


In [ ]:
# ─── Ejercicio A.5 — Parquet conserva los tipos y permite leer por columnas ──
# 1. Guarda en 'df_tipos' una COPIA de 'df_merged' en la que las columnas
#    'tipo_cuenta', 'ciudad' y 'tipo_contenido' sean categóricas.
# 2. Guarda 'df_tipos' en '../data/tipos_categoricos.parquet' y en
#    '../data/tipos_categoricos.csv' (ambos sin índice).
# 3. Lee los dos ficheros y guarda en 'tipo_desde_parquet' y
#    'tipo_desde_csv' el tipo de dato de la columna 'tipo_cuenta' en cada
#    tabla leída, escrito como texto.
#    Por ejemplo, uno de los dos valdría 'category' y el otro sería un tipo
#    de texto (razona cuál es cuál).
# 4. Guarda en 'df_dos_columnas' las columnas 'ciudad' y 'engagement_rate'
#    leyendo del Parquet únicamente esas dos columnas.
# 5. Escribe en un comentario qué ventaja aporta guardar las categóricas en
#    Parquet cuando el siguiente paso del proyecto (Bloque 5) vuelva a
#    leer este fichero.

df_tipos = None
tipo_desde_parquet = None
tipo_desde_csv = None
df_dos_columnas = None

# Tu razonamiento:


### Entrega opcional — `../scripts/pipeline_avanzada.py`

Si quieres consolidar los ejercicios avanzados como funciones reutilizables, implementa en `../scripts/pipeline_avanzada.py`:

| Función | Descripción |
|---|---|
| `analizar_categorias(df, columnas)` | Devuelve un DataFrame indexado por el nombre de cada columna con `tipo` (`category`), `categorias`, `memoria_antes_bytes`, `memoria_despues_bytes` y `ahorro_bytes`, sin modificar el original (ejercicio A.1) |
| `tabla_engagement_ciudad_contenido(df)` | Devuelve la tabla dinámica del engagement medio con `tipo_contenido` en las filas y `ciudad` en las columnas, con 0 en las combinaciones sin datos (ejercicio A.3) |
| `comparar_formatos(df, ruta_base)` | Guarda `<ruta_base>.csv` y `<ruta_base>.parquet` y devuelve un diccionario con `csv_bytes`, `parquet_bytes` y `ratio` (ejercicio A.4) |

Comprueba tu implementación con:

```bash
python -m pytest ../tests/test_bloque4_avanzado.py -v
```